# FiremeX — teach the model small flames (model-v5, Google Colab)

model-v4 knows two things: **smoke** and **fire**. This run adds a third class,
**small_flame** — a lighter or candle flame — from our EZVIZ camera clips at
close, mid and about 1 m distance.

**Why a separate class and not "fire".** A lighter is not a fire. If it were
called fire, every cigarette, candle or gas stove would set off the fire alarm.
As its own class the model can tell the two apart, and the add-on decides
what to do with it (by default: nothing — `small_flame` is not in
`hazard_classes`).

**Why it starts from model-v4.** Fine-tuning keeps what v4 learned about fire
and smoke; only the part that names the classes is rebuilt for three classes.
D-Fire and the Home Fire Dataset are trained again alongside our clips
(*replay*), otherwise the model forgets fire and smoke — that is what happened
to our first v3 attempt on the Mac. Our camera pictures are repeated 10 times
so the ~80 small-flame pictures are not lost among ~20,000 others.

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Runtime → **Run all**. Allow Google Drive, and upload
   **`firemex_own_dataset_v2.zip`** (project folder `ml/datasets/`) when the button appears.
3. Progress is saved to Google Drive every epoch: **if Colab disconnects, Run all again.**
4. The last cells check that fire and smoke are **not worse** than model-v4,
   report how many small flames it finds, and download the model.

In [ ]:
# Google Drive keeps the training progress safe if Colab disconnects.
from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/firemex_small_flame"
import os
os.makedirs(SAVE, exist_ok=True)

# YOLO26 needs Ultralytics 8.4. Pinned, so a re-run uses exactly the same code.
!pip -q install ultralytics==8.4.176 kagglehub onnx onnxslim
# wandb would hijack logging; dill makes model files the add-on cannot open.
!pip -q uninstall -y wandb dill
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to a T4 GPU runtime")

In [ ]:
# 1) Home Fire Dataset from its GitHub release. Its classes are 0 = flame, 1 = smoke;
#    FiremeX uses 0 = smoke, 1 = fire, so every label is swapped (once).
import glob, shutil
BASE = "https://github.com/PengBo0/Home-fire-dataset/releases/download/v1.0.0"
os.makedirs("/content/homefire", exist_ok=True)
for split in ("train", "val", "test"):
    if not os.path.exists(f"/content/homefire/{split}/images"):
        !wget -q -O /content/{split}.zip {BASE}/{split}.zip
        !unzip -q -o /content/{split}.zip -d /content/homefire/{split}
        os.remove(f"/content/{split}.zip")
SWAP = {"0": "1", "1": "0"}
if not os.path.exists("/content/homefire/.swapped"):
    for lab in glob.glob("/content/homefire/*/labels/*.txt"):
        lines = [l.split() for l in open(lab) if l.strip()]
        with open(lab, "w") as fh:
            for l in lines:
                fh.write(" ".join([SWAP[l[0]]] + l[1:]) + "\n")
    open("/content/homefire/.swapped", "w").close()
for split in ("train", "val", "test"):
    print("Home Fire", split, len(glob.glob(f"/content/homefire/{split}/images/*")))

In [ ]:
# 2) D-Fire, ALL of it (classes already 0 = smoke, 1 = fire). Used in place - no copying.
#    If Kaggle asks you to sign in: kaggle.com -> Settings -> Create New Token, upload
#    kaggle.json with the folder icon on the left, and run this cell again.
import kagglehub
root = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")
def split_dir(*names):
    for n in names:
        hits = glob.glob(f"{root}/**/{n}/images", recursive=True)
        if hits:
            return hits[0]
dfire = {"train": split_dir("train"), "val": split_dir("val", "valid"), "test": split_dir("test")}
for k, v in dfire.items():
    print("D-Fire", k, v, len(glob.glob(f"{v}/*")) if v else "MISSING")
empty = sum(1 for p in glob.glob(f"{os.path.dirname(dfire['train'])}/labels/*.txt") if os.path.getsize(p) == 0)
print("D-Fire training pictures with no fire or smoke (negatives):", empty)

In [ ]:
# 3) Our camera pictures (v2: lighter flames labelled small_flame). Upload
#    firemex_own_dataset_v2.zip when the button appears. Its test clips
#    (take3_02 burning paper, lighter_1m_02 lighter at mid and far) are never trained on.
from google.colab import files
if not os.path.exists("/content/own/train/images"):
    up = files.upload()
    zip_name = next(iter(up))
    !unzip -q -o "/content/{zip_name}" -d /content/own
print("own train", len(glob.glob("/content/own/train/images/*")),
      "| own test", len(glob.glob("/content/own/test/images/*")))

In [ ]:
# 4) One training list. Classes: 0 smoke, 1 fire (as D-Fire and Home Fire are labelled
#    after the swap) and 2 small_flame (only in our clips). Our camera's training
#    pictures are listed 10 times. Validation mixes indoor, outdoor and our camera.
import yaml
names = {0: "smoke", 1: "fire", 2: "small_flame"}
own_train = sorted(glob.glob("/content/own/train/images/*.jpg"))
with open("/content/own_x10.txt", "w") as fh:
    fh.write("\n".join(own_train * 10) + "\n")
train = ["/content/homefire/train/images", "/content/own_x10.txt", dfire["train"]]
val = ["/content/homefire/val/images", "/content/own/test/images"] + ([dfire["val"]] if dfire["val"] else [])
yaml.safe_dump({"train": train, "val": val, "names": names}, open("/content/mix.yaml", "w"))
tests = {"Home fire test": "/content/homefire/test/images",
         "Our camera test": "/content/own/test/images",
         "D-Fire test": dfire["test"]}
for label, path in tests.items():
    yaml.safe_dump({"train": path, "val": path, "names": names},
                   open(f"/content/test_{label.split()[0].lower()}.yaml", "w"))
print(open("/content/mix.yaml").read())
print("our camera training pictures:", len(own_train), "x 10")

In [ ]:
# 5) Fine-tune model-v4 for three classes. Ultralytics keeps every layer of v4 that
#    still fits and rebuilds only the class layer. 30 epochs: v4 already knows fire
#    and smoke. Same two augmentation choices as v4 (hue kept, no upside-down).
#    Progress is saved to Drive every epoch; a re-run resumes from it.
from ultralytics import YOLO
!wget -q -nc -O /content/model_v4.pt https://github.com/malika1234m/Firemax/releases/download/model-v4/fire_model.pt
RUN = f"{SAVE}/runs/v5"
last = f"{RUN}/weights/last.pt"
if os.path.exists(last):
    print("Resuming from", last)
    try:
        YOLO(last).train(resume=True)
    except AssertionError as exc:          # raised when every epoch is already done
        print("Training already finished:", exc)
else:
    YOLO("/content/model_v4.pt").train(
        data="/content/mix.yaml", epochs=30, patience=10, imgsz=640, batch=32,
        seed=0, deterministic=True, workers=2,
        hsv_h=0.005, flipud=0.0,
        project=f"{SAVE}/runs", name="v5", exist_ok=True)

In [ ]:
# 6) model-v5 vs model-v4 on every test set, per class. v4 has no small_flame class,
#    so fire and smoke are compared on their own; small_flame is v5's new skill.
import pandas as pd
def score(weights, label):
    r = YOLO(weights).val(data=f"/content/test_{label.split()[0].lower()}.yaml", split="val",
                         imgsz=640, batch=32, plots=False, verbose=False)
    row = {}
    for k, i in enumerate(r.box.ap_class_index):
        name = r.names[int(i)]
        row[f"{name} AP50"] = round(float(r.box.ap50[k]), 3)
        row[f"{name} recall"] = round(float(r.box.r[k]), 3)
    return row
candidates = {"model-v4": "/content/model_v4.pt",
              "v5 best.pt": f"{RUN}/weights/best.pt",
              "v5 last.pt": f"{RUN}/weights/last.pt"}
rows = [{"model": m, "test set": t, **score(w, t)} for m, w in candidates.items() for t in tests]
table = pd.DataFrame(rows)
table.to_csv(f"{SAVE}/comparison.csv", index=False)
display(table)

In [ ]:
# 7) Keep v5 only if fire and smoke are not more than 0.02 worse than v4 on any test set
#    (the reason we fine-tuned with replay) and it finds small flames on the held-out clip.
def cell(m, t, col):
    v = table[(table.model == m) & (table["test set"] == t)][col]
    return float(v.iloc[0]) if col in table and len(v) and pd.notna(v.iloc[0]) else None
def fire_smoke_ok(m):
    bad = {}
    for t in tests:
        for col in ("fire AP50", "smoke AP50"):
            old, new = cell("model-v4", t, col), cell(m, t, col)
            if old is not None and (new is None or new < old - 0.02):
                bad[f"{t} {col}"] = (old, new)
    return bad
new = max(["v5 best.pt", "v5 last.pt"],
          key=lambda m: (not fire_smoke_ok(m), cell(m, "Our camera test", "small_flame AP50") or 0))
worse = fire_smoke_ok(new)
sf_ap, sf_r = cell(new, "Our camera test", "small_flame AP50"), cell(new, "Our camera test", "small_flame recall")
print(f"{new}: small_flame on the held-out lighter clip  AP50 {sf_ap}  recall {sf_r}")
PASSED = not worse and (sf_ap or 0) >= 0.5
print("PASSED - downloading" if PASSED else f"NOT good enough - keep model-v4. Worse: {worse or 'small_flame AP50 < 0.5'}")

In [ ]:
# 8) Save the chosen weights, export ONNX, download both. Named *_candidate if the
#    gate failed - still worth testing on our videos.
tag = "fire_model_v5_small_flame" + ("" if PASSED else "_candidate")
shutil.copy(candidates[new], f"/content/{tag}.pt")
YOLO(f"/content/{tag}.pt").export(format="onnx", imgsz=640, simplify=True)   # writes /content/{tag}.onnx
for f in (f"/content/{tag}.pt", f"/content/{tag}.onnx"):
    shutil.copy(f, SAVE)
    !sha256sum "{f}"
    files.download(f)